# Harmony Project 3 — QLoRA Fine-Tuning (Qwen2.5-7B, 4-bit NF4)

**Run order:** Cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8. Cell 8 runs the full automated 7-run sweep.

**QLoRA vs LoRA notebook differences:**
- Base model loaded in 4-bit NF4 (~5–8 GB vs ~15 GB FP16) — fits on a single T4
- Optimizer: `paged_adamw_8bit` (pages optimizer states to CPU when needed)
- Full 7-run sweep fits in **one Kaggle session** (~8–9 hrs) because single-GPU is faster

**If a run crashes:** Restart kernel (Run → Restart) and re-run from Cell 1.

## Cell 1 — Install pinned dependencies

Same pins as the LoRA notebook for consistency. `bitsandbytes` is required for 4-bit NF4 quantization.

In [ ]:
!pip install -q \
  "transformers==4.46.3" \
  "trl==0.11.4" \
  peft bitsandbytes accelerate datasets wandb json-repair

## Cell 2 — Configuration

Only edit `BASE_MODEL_REVISION` if a newer HF Hub commit is desired. Everything else is locked.

In [ ]:
BASE_MODEL          = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
WANDB_PROJECT       = "harmony-p3-qlora"
SEED                = 42

TRAIN_PATH = "/kaggle/input/datasets/keertanaks2004/harmony-ade-data/train.jsonl"
VAL_PATH   = "/kaggle/input/datasets/keertanaks2004/harmony-ade-data/val.jsonl"

print("Config loaded.")

## Cell 3 — Pre-flight checks

Verifies GPU, data files, row counts, and JSONL format. Fix any assert failure before running Cell 8.

In [ ]:
import os, json, torch
from pathlib import Path

# 1. GPU
assert torch.cuda.is_available(), "No GPU — this notebook requires T4x2"
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

# 2. Data files
assert Path(TRAIN_PATH).exists(), f"MISSING: {TRAIN_PATH}"
assert Path(VAL_PATH).exists(),   f"MISSING: {VAL_PATH}"

# 3. Row counts
n_train = sum(1 for _ in open(TRAIN_PATH))
n_val   = sum(1 for _ in open(VAL_PATH))
print(f"Train rows: {n_train:,}   Val rows: {n_val:,}")
assert n_train > 15000, f"train.jsonl too small ({n_train})"
assert n_val   > 2000,  f"val.jsonl too small ({n_val})"

# 4. Format spot-check
with open(TRAIN_PATH) as f:
    s = json.loads(f.readline())
assert "messages" in s and "schema_version" in s["messages"][1]["content"]
print("Pre-flight passed.")

## Cell 4 — Load tokenizer ONLY

**Do not load the model here.** Cell 6 defines `load_fresh_model()` which loads the 4-bit model
inside `run_one()`. Loading the model here would leave it in VRAM and cause OOM when Cell 8 tries
to load it again.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Tokenizer loaded  (vocab={len(tokenizer):,})")

## Cell 5 — Load and format dataset

Applies the Qwen2.5 chat template. Actual token length after templating: p95 ≈ 552, max ≈ 565
(the instruction wrapper adds ~400 tokens). `max_seq_length=600` in Cell 8 covers all examples.

In [ ]:
import json
import numpy as np
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def format_example(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
    }

train_dataset = Dataset.from_list(load_jsonl(TRAIN_PATH)).map(format_example)
val_dataset   = Dataset.from_list(load_jsonl(VAL_PATH)).map(format_example)
print(f"Train: {len(train_dataset):,}   Val: {len(val_dataset):,}")

# Token length check (200-example sample)
lens = [len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"])
        for ex in train_dataset.select(range(200))]
print(f"Token length  p50={np.percentile(lens,50):.0f}  "
      f"p95={np.percentile(lens,95):.0f}  max={max(lens)}")
assert max(lens) <= 1024, f"Some examples > 1024 tokens: {max(lens)}"
print("Dataset ready.")

## Cell 6 — Helper definitions

Defines three helpers used by `run_one()` in Cell 7:

1. **Loss patch** — fixes a transformers 4.46.3 bug where `num_items_in_batch` lands on a
   different device than `loss` during multi-GPU training.
2. **DataCollatorForCompletionOnlyLM** — TRL 0.11.4 removed the built-in version. This custom
   implementation masks loss on all tokens except the assistant turn.
3. **`load_fresh_model()`** — loads the base model in 4-bit NF4, calls
   `prepare_model_for_kbit_training()`, and sets `use_cache=False`. Called fresh at the start
   of every run to guarantee a clean GPU state.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# ── 1. Loss patch (transformers 4.46.3 multi-GPU device mismatch) ────────────
import transformers.loss.loss_utils as _loss_utils

def _patched_xent(source, target, num_items_in_batch=None, ignore_index=-100, **kwargs):
    reduction = "sum" if num_items_in_batch is not None else "mean"
    loss = torch.nn.functional.cross_entropy(
        source, target, ignore_index=ignore_index, reduction=reduction)
    if reduction == "sum":
        if hasattr(num_items_in_batch, "to"):
            num_items_in_batch = num_items_in_batch.to(loss.device)
        loss = loss / num_items_in_batch
    return loss

_loss_utils.fixed_cross_entropy = _patched_xent
print("Loss patch applied.")

# ── 2. DataCollator (TRL 0.11.4 removed the built-in version) ───────────────
class DataCollatorForCompletionOnlyLM:
    def __init__(self, response_template, tokenizer):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(
            response_template, add_special_tokens=False)

    def __call__(self, instances):
        input_ids = [torch.tensor(inst["input_ids"]) for inst in instances]
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True,
            padding_value=self.tokenizer.pad_token_id)
        labels = input_ids.clone()
        tpl = self.response_template_ids
        for i in range(len(labels)):
            seq = labels[i].tolist()
            found = False
            for j in range(len(seq) - len(tpl) + 1):
                if seq[j: j + len(tpl)] == tpl:
                    labels[i, : j + len(tpl)] = -100
                    found = True
                    break
            if not found:
                labels[i, :] = -100
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != self.tokenizer.pad_token_id).long(),
            "labels": labels,
        }

print("DataCollator defined.")

# ── 3. Model loader — 4-bit NF4, single GPU (fits on one T4) ────────────────
# QLoRA 4-bit model is ~5-8 GB vs ~15 GB for FP16 LoRA — fits on one T4 alone.
# device_map="auto" puts everything on GPU 0. No balanced split needed.
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # float16, NOT bfloat16 — T4 has no native BF16
)

def load_fresh_model():
    gc.collect()
    torch.cuda.empty_cache()
    m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        revision=BASE_MODEL_REVISION,
        quantization_config=BNB_CONFIG,
        device_map="auto",               # 4-bit fits on GPU 0 alone — no cross-GPU issues
    )
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m)   # enables grad checkpointing + FP32 layer norms
    return m

print("load_fresh_model() defined.")
print("All helpers ready.")

## Cell 7 — `run_one()` — single training run definition

Defines one complete training run: load model → configure LoRA → train → save (if final run) → free VRAM.

The `try/finally` block guarantees the model is freed from VRAM even if training crashes.
This prevents the "model stuck in VRAM" OOM bug when the sweep moves to the next run.

**QLoRA-specific vs LoRA notebook:**
- `optim="paged_adamw_8bit"` — pages optimizer states to CPU under memory pressure
- `fp16=False, bf16=False` — precision is controlled by `BitsAndBytesConfig`, not trainer flags

In [ ]:
import json, wandb
from pathlib import Path
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

def run_one(run_name, learning_rate, lora_rank,
            max_steps=500, num_epochs=3, save_adapter=False):

    lora_alpha = lora_rank * 2
    out_dir    = f"/kaggle/working/checkpoints/{run_name}"
    model, trainer = None, None

    print(f"\n{'='*62}")
    print(f"  RUN  : {run_name}")
    print(f"  lr={learning_rate}   rank={lora_rank}   alpha={lora_alpha}")
    print(f"{'='*62}")

    try:
        # ── Load fresh 4-bit model ───────────────────────────────────────────
        model = load_fresh_model()
        print(f"  GPU 0 after load: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free")

        # ── LoRA config ──────────────────────────────────────────────────────
        lora_cfg = LoraConfig(
            r=lora_rank, lora_alpha=lora_alpha,
            target_modules=["q_proj","k_proj","v_proj","o_proj",
                            "gate_proj","up_proj","down_proj"],
            lora_dropout=0.05, bias="none",
            task_type=TaskType.CAUSAL_LM,
        )

        # ── Data collator ────────────────────────────────────────────────────
        collator = DataCollatorForCompletionOnlyLM(
            response_template="<|im_start|>assistant\n",
            tokenizer=tokenizer,
        )

        # ── W&B ─────────────────────────────────────────────────────────────
        wandb.init(
            project=WANDB_PROJECT, name=run_name, reinit=True,
            config={"lr": learning_rate, "rank": lora_rank, "alpha": lora_alpha,
                    "max_steps": max_steps, "num_epochs": num_epochs,
                    "base_model": BASE_MODEL, "revision": BASE_MODEL_REVISION,
                    "method": "QLoRA-4bit-NF4"},
        )

        # ── Auto-resume if this run was interrupted ──────────────────────────
        resume_ckpt = None
        if Path(out_dir).exists():
            ckpts = sorted(Path(out_dir).glob("checkpoint-*"),
                           key=lambda p: int(p.name.split("-")[-1]))
            if ckpts:
                resume_ckpt = str(ckpts[-1])
                print(f"  Resuming from {resume_ckpt}")

        # ── Training config ──────────────────────────────────────────────────
        sft_config = SFTConfig(
            output_dir=out_dir,
            num_train_epochs=num_epochs if max_steps == -1 else 9999,
            max_steps=max_steps if max_steps != -1 else -1,

            # batch
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=8,       # effective batch = 16
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},

            # optimizer — paged_adamw_8bit pages states to CPU (QLoRA requirement)
            optim="paged_adamw_8bit",
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.0,

            # precision — controlled by BitsAndBytesConfig, NOT trainer flags
            fp16=False,
            bf16=False,
            average_tokens_across_devices=False,

            # sequence
            max_seq_length=600,                  # actual max after chat template = 565
            dataset_text_field="text",

            # checkpointing
            eval_strategy="steps", eval_steps=100,
            save_strategy="steps", save_steps=100, save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", greater_is_better=False,

            # logging
            logging_steps=10,
            report_to="wandb", run_name=run_name,
            seed=SEED, data_seed=SEED,
        )

        # ── Train ────────────────────────────────────────────────────────────
        trainer = SFTTrainer(
            model=model, args=sft_config,
            train_dataset=train_dataset, eval_dataset=val_dataset,
            data_collator=collator, peft_config=lora_cfg,
            tokenizer=tokenizer,                 # trl 0.11.4 still uses this name
        )

        trainer.train(resume_from_checkpoint=resume_ckpt)
        best_eval_loss = trainer.state.best_metric
        print(f"  {run_name}  eval_loss={best_eval_loss:.4f}")

        # ── Save adapter (final run only) ────────────────────────────────────
        if save_adapter:
            adapter_dir = "/kaggle/working/adapter"
            trainer.model.save_pretrained(adapter_dir)
            tokenizer.save_pretrained(adapter_dir)
            meta = {
                "method": "QLoRA",
                "run_name": run_name,
                "base_model": BASE_MODEL,
                "base_model_revision": BASE_MODEL_REVISION,
                "quantization": "4-bit NF4 double-quant, compute_dtype=float16",
                "learning_rate": learning_rate,
                "lora_rank": lora_rank,
                "lora_alpha": lora_alpha,
                "lora_dropout": 0.05,
                "best_eval_loss": round(float(best_eval_loss), 6),
                "num_epochs": num_epochs,
                "max_seq_length": 600,
                "per_device_train_batch_size": 2,
                "gradient_accumulation_steps": 8,
                "effective_batch_size": 16,
                "optimizer": "paged_adamw_8bit",
                "fp16": False, "bf16": False,
                "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                                   "gate_proj","up_proj","down_proj"],
            }
            with open(f"{adapter_dir}/training_args.json", "w") as f:
                json.dump(meta, f, indent=2)
            print(f"  Adapter saved to {adapter_dir}")

        wandb.finish()
        return best_eval_loss

    finally:
        # ── Guaranteed VRAM cleanup — runs even on exception ─────────────────
        if trainer is not None: del trainer
        if model   is not None: del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  Cleanup done  GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free")

print("run_one() defined and ready.")

## Cell 8 — Automated 7-run sweep (RUN AND LEAVE)

Runs all 7 QLoRA training jobs sequentially. Results logged to W&B project `harmony-p3-qlora`.

| Phase | Runs | What changes |
|---|---|---|
| 1 — LR sweep | 3 × 500 steps | rank=16 fixed, LR ∈ {1e-4, 2e-4, 5e-4} |
| 2 — Rank sweep | 3 × 500 steps | best_LR fixed, rank ∈ {8, 16, 32} |
| 3 — Final | 3 epochs | best_LR + best_rank, saves adapter |

**ETA: ~8–9 hours total** (single-GPU 4-bit is faster than dual-GPU FP16 LoRA).
The full sweep fits in one 12-hour Kaggle session.

Add `WANDB_API_KEY` as a Kaggle Secret before running.

In [ ]:
import os, gc, torch
from kaggle_secrets import UserSecretsClient

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_API_KEY"]      = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_SILENT"]       = "true"

# ── Safety: GPU must be empty before starting ─────────────────────────────────
for _v in ["model", "trainer"]:
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

print(f"GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free  |  "
      f"GPU 1: {torch.cuda.mem_get_info(1)[0]/1e9:.1f} GB free")
assert torch.cuda.mem_get_info(0)[0] > 14e9, (
    "GPU 0 not empty — restart kernel (Run → Restart) and re-run from Cell 1."
)

# ════════════════════════════════════════════════════════════════════════════
# PHASE 1 — LR sweep  (rank=16 fixed, 500 steps each)
# ════════════════════════════════════════════════════════════════════════════
lr_sweep = {
    "qlora_lr_1e4_r16": 1e-4,
    "qlora_lr_2e4_r16": 2e-4,
    "qlora_lr_5e4_r16": 5e-4,
}
lr_results = {}
for name, lr in lr_sweep.items():
    lr_results[lr] = run_one(name, lr, lora_rank=16, max_steps=500)

best_lr = min(lr_results, key=lr_results.get)
print(f"\nBest LR : {best_lr}  (eval_loss={lr_results[best_lr]:.4f})")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 2 — Rank sweep  (best_lr fixed, 500 steps each)
# ════════════════════════════════════════════════════════════════════════════
rank_sweep = {
    "qlora_bestlr_r8":  8,
    "qlora_bestlr_r16": 16,
    "qlora_bestlr_r32": 32,
}
rank_results = {}
for name, rank in rank_sweep.items():
    rank_results[rank] = run_one(name, best_lr, lora_rank=rank, max_steps=500)

best_rank = min(rank_results, key=rank_results.get)
print(f"\nBest Rank : {best_rank}  (eval_loss={rank_results[best_rank]:.4f})")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 3 — Final full run  (3 epochs, saves adapter)
# ════════════════════════════════════════════════════════════════════════════
final_loss = run_one(
    run_name="qlora_final",
    learning_rate=best_lr,
    lora_rank=best_rank,
    max_steps=-1,
    num_epochs=3,
    save_adapter=True,
)

print(f"\n{'='*62}")
print(f"  SWEEP COMPLETE")
print(f"  best_lr   = {best_lr}")
print(f"  best_rank = {best_rank}")
print(f"  final eval_loss = {final_loss:.4f}")
print(f"  Adapter saved to /kaggle/working/adapter/")
print(f"{'='*62}")
print("\nDownload /kaggle/working/adapter/ as a zip before session ends.")

## Done

When Cell 8 finishes:

1. **Download** `/kaggle/working/adapter/` as a zip (right-click in the Output panel → Download)
2. **Check W&B** at https://wandb.ai → project `harmony-p3-qlora` for all run metrics
3. **Commit** the adapter zip to `models/adapters/qlora_v1/` in the Harmony repo

If the session hits the 12-hour limit mid-sweep:
- Checkpoints persist in `/kaggle/working/checkpoints/{run_name}/`
- Re-run cells 1 → 8 — Cell 8 auto-resumes the interrupted run from its last checkpoint
- Completed runs re-validate quickly (~30 sec) before the sweep continues